In [ ]:
%load_ext autoreload
%autoreload 2

**Author:** Salvador Navas  
**Date:** 2025-06-27

### GloFAS — Global Flood Awareness System (Copernicus EWDS)

**GloFAS** (Global Flood Awareness System) is a global hydrological forecasting and monitoring system developed jointly by the European Commission and ECMWF. It uses the **LISFLOOD** hydrological model forced with ERA5 reanalysis to produce daily river discharge estimates worldwide.

#### 💡 **Key characteristics:**

- **Provider:** Copernicus Emergency Management Service (CEMS), operated by ECMWF.
- **Dataset:** `cems-glofas-historical` — consolidated reanalysis v4.0.
- **Hydrological model:** LISFLOOD forced with ERA5 reanalysis.
- **Variable:** River discharge in the last 24 hours (`dis24`), in m³/s.
- **Spatial resolution:** 0.05° (~5 km at the equator).
- **Temporal resolution:** Daily.
- **Temporal coverage:** 1979 – present.

#### 🖥️ **Advantages of GloFAS reanalysis:**

1. **Global coverage:** Continuous daily discharge anywhere in the world, even in ungauged basins.
2. **Long record:** More than 40 years of consistent reanalysis data.
3. **High resolution:** 0.05° grid resolves most major rivers.
4. **Free access:** Available via the Copernicus EWDS API with a free account.

#### ⚙️ **Prerequisites:**

1. Create a free account at [https://ewds.climate.copernicus.eu](https://ewds.climate.copernicus.eu)
2. Accept the terms for `cems-glofas-historical` on the portal.
3. Configure `~/.cdsapirc`:

```ini
url: https://ewds.climate.copernicus.eu/api
key: <your-personal-api-key>
```

4. Install required packages: `pip install cdsapi xarray netCDF4`

#### 🔗 **Access and download:**

- **EWDS portal:** [https://ewds.climate.copernicus.eu](https://ewds.climate.copernicus.eu)
- **Dataset page:** [https://ewds.climate.copernicus.eu/datasets/cems-glofas-historical](https://ewds.climate.copernicus.eu/datasets/cems-glofas-historical)

In [ ]:
from pyhydra.data_sources.river_discharge import (
    download_glofas,
    download_glofas_by_year,
    read_glofas_nc,
)
from pyhydra.utils import interactive_map
import matplotlib.pyplot as plt
import pandas as pd
import os
from pathlib import Path

## 🗺️ Select area of interest

Draw a **rectangle** on the map to define the download region. Read `coordinates_list[0]` in the next cell.

In [ ]:
coordinates_list = interactive_map(zoom=4, center=(40, -5))

In [ ]:
# === CONFIGURATION ===
API_URL = 'https://ewds.climate.copernicus.eu/api'
API_KEY = 'YOUR_EWDS_API_KEY'   # or None to use ~/.cdsapirc
# Get your key at https://ewds.climate.copernicus.eu/profile

AREA = coordinates_list[0] if coordinates_list else [44, -10, 35, 5]  # [N, W, S, E]
YEARS = list(range(2000, 2021))
OUTPUT_DIR = '/workspace/data/glofas/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

SYSTEM_VERSION = 'version_4_0'
PRODUCT_TYPE   = 'consolidated'
FILE_FORMAT    = 'netcdf4'
MONTHS         = list(range(1, 13))

LAT = 39.5
LON = -3.5

print(f'Area (N, W, S, E): {AREA}')

---
## 1. ⬇️ Download — full range (single request)

Submits one CDS request covering all years at once. Suitable for short periods; may time out for large date ranges.

In [ ]:
paths = download_glofas(
    area=AREA,
    years=YEARS,
    output_dir=OUTPUT_DIR,
    system_version=SYSTEM_VERSION,
    product_type=PRODUCT_TYPE,
    months=MONTHS,
    file_format=FILE_FORMAT,
    api_key=API_KEY,
    api_url=API_URL,
)
print('Downloaded:', paths)

---
## 2. ⬇️ Download — year by year (recommended)

Submits one CDS request per year. Avoids timeout issues with large requests and allows partial downloads to resume.

In [ ]:
paths = download_glofas_by_year(
    area=AREA,
    years=YEARS,
    output_dir=OUTPUT_DIR,
    system_version=SYSTEM_VERSION,
    product_type=PRODUCT_TYPE,
    months=MONTHS,
    file_format=FILE_FORMAT,
    api_key=API_KEY,
    api_url=API_URL,
)
print(f'Downloaded {len(paths)} files:')
for p in paths:
    print(' ', p)

---
## 3. 📖 Reading downloaded NetCDF files

In [ ]:
# Extract discharge at a specific coordinate from a single .nc file
NC_FILE = f'{OUTPUT_DIR}/glofas_cems-glofas-historical_2000.nc'

df_glofas = read_glofas_nc(NC_FILE, lat=LAT, lon=LON)
df_glofas = df_glofas.set_index('date')
print(df_glofas.describe())
df_glofas.head()

In [ ]:
# Load and concatenate multiple yearly files into a single series
dfs = []
for year in YEARS:
    nc = f'{OUTPUT_DIR}/glofas_cems-glofas-historical_{year}.nc'
    if Path(nc).exists():
        dfs.append(read_glofas_nc(nc, lat=LAT, lon=LON))

df_all = pd.concat(dfs).set_index('date').sort_index()
print(f'Period : {df_all.index[0].date()} → {df_all.index[-1].date()}')
print(f'Mean Q : {df_all["discharge"].mean():.2f} m³/s')
print(f'Max  Q : {df_all["discharge"].max():.1f} m³/s')
df_all.head()

---
## 4. 📊 Visualisation

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

axes[0].plot(df_all.index, df_all['discharge'], lw=0.6, color='darkorange')
axes[0].set_ylabel('GloFAS discharge (m³/s)')
axes[0].set_title(f'GloFAS v4.0 · ({LAT}°N, {LON}°E)', fontsize=13)
axes[0].grid(alpha=0.3)

monthly = df_all['discharge'].groupby(df_all.index.month).mean()
axes[1].bar(monthly.index, monthly.values, color='darkorange', alpha=0.85)
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                          'Jul','Aug','Sep','Oct','Nov','Dec'])
axes[1].set_ylabel('Mean discharge (m³/s)')
axes[1].set_title('Seasonal regime')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()